In [2]:
import pandas as pd
import requests
import sqlite3
import logging

logging.basicConfig(
    filename="posts.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

API_URL = "https://jsonplaceholder.typicode.com/posts"
CSV_PATH = "clean_posts.csv"
DB_PATH = "posts.db"

In [3]:
def fetch_posts(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # raises error for bad status codes
        logging.info("Data fetched successfully from API")
        return response.json()
    except requests.exceptions.RequestException as e:
        logging.error(f"Error fetching data: {e}")
        return []

# Fetch data
data = fetch_posts(API_URL)

print(f"Total records fetched: {len(data)}")

Total records fetched: 100


In [4]:
try:
    df = pd.DataFrame(data)
    logging.info("Data loaded into DataFrame successfully")
except Exception as e:
    logging.error(f"Error creating DataFrame: {e}")
    df = pd.DataFrame()

df.head()

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [5]:
try:
    # Keep only required columns
    df = df[["userId", "id", "title", "body"]]

    # Word count in title
    df["word_count"] = df["title"].astype(str).str.split().str.len()

    # Filter condition
    df = df[df["word_count"] >= 4]

    # Standardization
    df["title"] = df["title"].astype(str).str.strip().str.title()
    df["body"] = df["body"].astype(str).str.strip()

    logging.info("Data transformation completed successfully")

except Exception as e:
    logging.error(f"Error during transformation: {e}")

In [6]:
df['body']

0     quia et suscipit\nsuscipit recusandae consequu...
2     et iusto sed quo iure\nvoluptatem occaecati om...
3     ullam et saepe reiciendis voluptatem adipisci\...
5     ut aspernatur corporis harum nihil quis provid...
7     dignissimos aperiam dolorem qui eum\nfacilis q...
                            ...                        
94    earum voluptatem facere provident blanditiis v...
95    in non odio excepturi sint eum\nlabore volupta...
96    eum non blanditiis soluta porro quibusdam volu...
98    quo deleniti praesentium dicta non quod\naut e...
99    cupiditate quo est a modi nesciunt soluta\nips...
Name: body, Length: 89, dtype: str

In [7]:
try:
    total_posts = len(data)
    filtered_posts = len(df)

    print(f"Total posts fetched: {total_posts}")
    print(f"Posts after filtering: {filtered_posts}")

    # Top 3 users by post count
    top_users = df["userId"].value_counts().head(3)

    print("\nTop 3 users by post count:")
    print(top_users)

except Exception as e:
    logging.error(f"Error in analysis step: {e}")

Total posts fetched: 100
Posts after filtering: 89

Top 3 users by post count:
userId
3    10
6    10
8    10
Name: count, dtype: int64


In [8]:
try:
    df.to_csv(CSV_PATH, index=False)
    logging.info(f"Data saved to CSV at {CSV_PATH}")
except Exception as e:
    logging.error(f"Error saving CSV: {e}")

In [9]:
try:
    conn = sqlite3.connect(DB_PATH)

    df.to_sql("posts", conn, if_exists="replace", index=False)

    conn.close()

    logging.info(f"Data saved to SQLite database at {DB_PATH}")

except Exception as e:
    logging.error(f"Error saving to SQLite: {e}")